NAME: Minal Honali Raghunandan

Student ID: 6908107

Notebook 5: Ablation Study and Statistical validation

Tests which acoustic feature families drive valence and arousal models on isolated grps, tells whether the perfoermance gap between the two targets is statistically significant or not


In [ ]:
import os, re, joblib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import KFold, cross_val_score
from scipy import stats

from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/Dissertation/data/processed"
MODEL_DIR = "/content/drive/MyDrive/Dissertation/models"


X_train = pd.read_csv(os.path.join(DATA_PATH, "X_train.csv"))
y_train = pd.read_csv(os.path.join(DATA_PATH, "y_train.csv"))
trained_cols = joblib.load(os.path.join(MODEL_DIR, "feature_columns.pkl"))

regx = re.compile(r"\[|\]|<")
X_train.columns = [regx.sub("_",c) for c in X_train.columns]
print(X_train.shape, y_train.shape)

Mounted at /content/drive
(1395, 302) (1395, 2)


In [ ]:
FAMILIES = {
    "loudness_energy": ["audspec_lengthL1norm", "audspecRasta_lengthL1norm",
                        "pcm_RMSenergy", "audSpec_Rfilt"],
    "timbral_mfcc":    ["mfcc"],
    "spectral_shape":  ["spectralVariance", "spectralKurtosis", "spectralSkewness",
                        "spectralCentroid", "spectralRollOff", "spectralSlope",
                        "spectralEntropy", "spectralHarmonicity", "spectralFlux"],
    "voice_quality":   ["jitter", "shimmer", "logHNR", "voicingFinalUnclipped"],
    "pitch":           ["F0final"],
}

def family_cols(keys):
    return [c for c in X_train.columns if any(k.lower() in c.lower() for k in keys)]

for name, keys in FAMILIES.items():
    cols = family_cols(keys)
    print(f"{name:20s} {len(cols):4d} features")

covered = set()
for keys in FAMILIES.values():
    covered |= set(family_cols(keys))
print(f"\nCovered: {len(covered)}/{len(X_train.columns)}")
print("Uncovered:", [c for c in X_train.columns if c not in covered][:10])

loudness_energy        64 features
timbral_mfcc          112 features
spectral_shape         63 features
voice_quality          39 features
pitch                   7 features

Covered: 285/302
Uncovered: ['pcm_zcr_sma_stddev_mean', 'pcm_zcr_sma_stddev_std', 'pcm_zcr_sma_amean_mean', 'pcm_zcr_sma_amean_std', 'pcm_fftMag_fband250-650_sma_stddev_mean', 'pcm_fftMag_fband250-650_sma_stddev_std', 'pcm_fftMag_fband250-650_sma_amean_mean', 'pcm_fftMag_fband250-650_sma_amean_std', 'pcm_fftMag_fband1000-4000_sma_stddev_mean', 'pcm_fftMag_fband1000-4000_sma_stddev_std']


In [ ]:
FAMILIES["spectral_shape"] += ["pcm_zcr", "fband"]

for name, keys in FAMILIES.items():
    print(f"{name:20s} {len(family_cols(keys)):4d} features")

covered = set()
for keys in FAMILIES.values():
    covered |= set(family_cols(keys))
print(f"\nCovered: {len(covered)}/{len(X_train.columns)}")

loudness_energy        64 features
timbral_mfcc          112 features
spectral_shape         80 features
voice_quality          39 features
pitch                   7 features

Covered: 302/302


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
targets = {"Valence": 0, "Arousal": 1}

def cv_r2(cols, target_idx):
    model = xgb.XGBRegressor(n_estimators=300, max_depth=4,
                             learning_rate=0.1, random_state=42)
    scores = cross_val_score(model, X_train[cols], y_train.iloc[:, target_idx],
                             cv=kf, scoring="r2")
    return scores

results = []
fold_scores = {}

# Full model baseline
for tname, tidx in targets.items():
    s = cv_r2(list(X_train.columns), tidx)
    fold_scores[("all_features", tname)] = s
    results.append({"Features": "all_features", "n": X_train.shape[1],
                    "Target": tname, "R2_mean": round(s.mean(), 4),
                    "R2_std": round(s.std(), 4)})
    print(f"all_features {tname}: {s.mean():.4f}")

# Each family alone
for fam, keys in FAMILIES.items():
    cols = family_cols(keys)
    for tname, tidx in targets.items():
        s = cv_r2(cols, tidx)
        fold_scores[(fam, tname)] = s
        results.append({"Features": fam, "n": len(cols), "Target": tname,
                        "R2_mean": round(s.mean(), 4), "R2_std": round(s.std(), 4)})
        print(f"{fam} {tname}: {s.mean():.4f}")

df_abl = pd.DataFrame(results)
print()
print(df_abl.pivot(index=["Features", "n"], columns="Target", values="R2_mean"))

all_features Valence: 0.5063
all_features Arousal: 0.5736
loudness_energy Valence: 0.4441
loudness_energy Arousal: 0.4847
timbral_mfcc Valence: 0.3915
timbral_mfcc Arousal: 0.4722
spectral_shape Valence: 0.4216
spectral_shape Arousal: 0.4881
voice_quality Valence: 0.2815
voice_quality Arousal: 0.3697
pitch Valence: 0.1595
pitch Arousal: 0.2004

Target               Arousal  Valence
Features        n                    
all_features    302   0.5736   0.5063
loudness_energy 64    0.4847   0.4441
pitch           7     0.2004   0.1595
spectral_shape  80    0.4881   0.4216
timbral_mfcc    112   0.4722   0.3915
voice_quality   39    0.3697   0.2815


### Ablation Findings

**No single feature family reproduces full-model performance.** The complete 302-feature
model achieves R² of 0.5736 for arousal and 0.5063 for valence. The strongest individual
family reaches 0.4881 and 0.4441 respectively roughly 85% of full performance for
arousal and 88% for valence. The remaining gain comes from combining families, indicating
that the model exploits complementary information across descriptor types rather than
relying on one dominant signal.

**Loudness and energy are the most efficient predictors.** With 64 features (21% of the
total) this family reaches 0.4847 for arousal 84% of full-model performance. Pe
feature, it is the most informative group by a clear margin, consistent with the SHAP
attributions in Notebook 4 where loudness descriptors dominated the global importance
ranking for arousal.

**Spectral shape marginally outperforms loudness for arousal, but with more features.**
Spectral descriptors reach 0.4881 for arousal against loudness at 0.4847, using 80
features rather than 64. This qualifies the SHAP interpretation: while individual
loudness features carry the highest attributions, spectral descriptors collectively
carry comparable information. The two accounts are compatible SHAP measures per-feature
contribution within the full model, whereas ablation measures the standalone predictive
capacity of a group.

**Timbral features underperform relative to their number.** MFCCs constitute the largest
family at 112 features but achieve only 0.4722 for arousal and 0.3915 for valence,
ranking third of five. This supports the interpretation advanced in Chapter 4 that
valence draws weakly on many timbral descriptors rather than strongly on a few: the
family is informative in aggregate but no individual member carries substantial signal.

**Voice quality and pitch contribute least.** Jitter, shimmer and harmonics-to-noise
ratio reach 0.3697 and 0.2815; pitch, with only 7 features, reaches 0.2004 and 0.1595.
Both nonetheless perform well above chance, and their inclusion in the SHAP rankings as
secondary contributors suggests they add refinement rather than primary signal.

**The arousal valence asymmetry holds across every feature family.** Arousal is predicted
more accurately than valence in all five ablations, with the gap ranging from 0.04
(pitch) to 0.10 (timbral MFCC). This establishes that the asymmetry is a property of the
targets rather than an artefact of any particular feature group no subset of the
acoustic representation predicts valence as well as it predicts arousal.